# Volatilitätsprognose: Chronos-2 vs. HAR vs. GARCH (Prototyp)

Dieses Notebook ist die Grundlage für das Gateway-Projekt:

1. **Daten:** 5-Minuten-Kerzen von Binance (kostenlos, ohne API-Key) → tägliche realisierte Varianz (RV) und Tagesrenditen.
2. **Experten** mit einheitlicher Schnittstelle: Chronos-2 (zero-shot), log-HAR, GARCH(1,1)-t und ein Random Walk als Baseline.
3. **Rolling-Origin-Forecasts** für jeden Tag und Horizont. Alle Forecasts (Mittelwert + 21 Quantile) werden als Parquet gespeichert. Kombinationen und das Gateway-Training laufen später rein offline auf diesen Dateien.
4. **Evaluation** auf der Validierungsperiode: QLIKE, Quantil-Loss, Intervall-Abdeckung, Diebold-Mariano.
5. **Oracle-Analyse:** Wie viel könnte ein perfektes Gateway überhaupt gewinnen?

**Splits (nach Zieldatum):** `train` → Gateway-Training, `val` → Modellauswahl und Tuning, `test` → **gesperrt** bis zur finalen Evaluation.

**Laufzeit:** RW, HAR und GARCH brauchen je Reihe typischerweise unter einer Minute. Chronos-2 läuft auf GPU in wenigen Minuten, auf CPU wird automatisch das kleinere `chronos-2-small` genutzt. Alle Ergebnisse werden gecacht: Ein neues Modell anhängen heißt, eine neue Expert-Klasse zu schreiben und nur diese laufen zu lassen.

In [ ]:
# Einmalig installieren (z.B. in Colab):
# %pip install -q chronos-forecasting arch "pandas[pyarrow]" matplotlib requests tqdm scipy

In [ ]:
import time, zipfile, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests
import torch
from arch import arch_model
from scipy.stats import norm
from tqdm.auto import tqdm

warnings.filterwarnings("ignore", module="arch")

# ---- Daten ----
SYMBOLS = ["BTCUSDT", "ETHUSDT", "SOLUSDT"]
KLINE_INTERVAL = "5m"
FIRST_MONTH = "2020-01"
LAST_MONTH = "2026-08"          # letzter vollständiger Monat
MIN_BARS_PER_DAY = 270          # von 288 möglichen 5-Min-Bars, sonst wird der Tag verworfen
DATA_DIR, RESULTS_DIR = Path("data"), Path("results")

# ---- Forecast-Setup ----
HORIZONS = [1, 5]               # Schritte (Tage) voraus
QUANTILES = [0.01, 0.05, 0.1, 0.15, 0.2, 0.25, 0.3, 0.35, 0.4, 0.45, 0.5,
             0.55, 0.6, 0.65, 0.7, 0.75, 0.8, 0.85, 0.9, 0.95, 0.99]   # = native Chronos-2-Quantile
QCOLS = [f"q{q:.2f}" for q in QUANTILES]
MIN_HISTORY = 500               # Tage Historie vor dem ersten Forecast-Ursprung
EST_WINDOW = 1000               # rollierendes Schätzfenster für RW / HAR / GARCH
GARCH_REFIT_EVERY = 5           # GARCH alle k Ursprünge neu schätzen, dazwischen Parameter fix

# ---- Splits (nach Zieldatum) ----
TRAIN_END = "2023-12-31"
VAL_END = "2024-12-31"
ALLOW_TEST = False              # erst ganz am Ende auf True setzen!

# ---- Chronos-2 ----
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
CHRONOS_MODEL = "amazon/chronos-2" if DEVICE == "cuda" else "autogluon/chronos-2-small"
CHRONOS_CONTEXT = 1024          # Tage Kontext (Chronos-2 kann bis 8192)
CHRONOS_BATCH = 128

In [ ]:
DATA_DIR.mkdir(exist_ok=True)
RESULTS_DIR.mkdir(exist_ok=True)
print("Device:", DEVICE, "| Chronos-Modell:", CHRONOS_MODEL)

## 1. Daten: Binance-5-Min-Kerzen → tägliche RV

Für jeden UTC-Tag $d$ ist die realisierte Varianz $RV_d = \sum_i r_{d,i}^2$ mit 5-Minuten-Logrenditen $r_{d,i}$, die Tagesrendite ist $\sum_i r_{d,i}$. Krypto handelt rund um die Uhr, es gibt also keine Overnight-Lücke. Tage mit zu vielen fehlenden Bars werden verworfen. Hinweis: Binance-Spotdaten haben ab 2025 Zeitstempel in Mikrosekunden statt Millisekunden, der Parser erkennt das automatisch.

In [ ]:
BASE_URL = "https://data.binance.vision/data/spot/monthly/klines/{sym}/{itv}/{sym}-{itv}-{ym}.zip"
KLINE_COLS = ["open_time", "open", "high", "low", "close", "volume", "close_time",
              "quote_volume", "n_trades", "taker_base", "taker_quote", "ignore"]


def download_month(sym, ym, itv=KLINE_INTERVAL):
    # Lädt eine Monatsdatei (mit Cache). None, falls es sie nicht gibt (z.B. Coin noch nicht gelistet).
    path = DATA_DIR / "raw" / sym / f"{sym}-{itv}-{ym}.zip"
    if path.exists():
        return path
    path.parent.mkdir(parents=True, exist_ok=True)
    r = requests.get(BASE_URL.format(sym=sym, itv=itv, ym=ym), timeout=60)
    if r.status_code in (403, 404):                  # Datei existiert nicht
        return None
    r.raise_for_status()
    path.write_bytes(r.content)
    return path


def read_kline_zip(path):
    with zipfile.ZipFile(path) as zf, zf.open(zf.namelist()[0]) as f:
        df = pd.read_csv(f, header=None)
    if not str(df.iloc[0, 0]).strip().isdigit():     # manche Dateien haben eine Kopfzeile
        df = df.iloc[1:]
    df = df.iloc[:, :len(KLINE_COLS)]
    df.columns = KLINE_COLS
    t = pd.to_numeric(df["open_time"]).astype("int64").to_numpy()
    unit = "us" if t.max() > 1e14 else "ms"          # ab 2025: Mikrosekunden
    return pd.Series(pd.to_numeric(df["close"]).to_numpy(float),
                     index=pd.to_datetime(t, unit=unit, utc=True), name="close")


def load_close(sym):
    months = pd.period_range(FIRST_MONTH, LAST_MONTH, freq="M").strftime("%Y-%m")
    parts = [read_kline_zip(p) for ym in tqdm(months, desc=sym, leave=False)
             if (p := download_month(sym, ym)) is not None]
    if not parts:
        raise RuntimeError(f"Keine Daten für {sym} gefunden. Symbol und Netzwerkzugang prüfen.")
    close = pd.concat(parts).sort_index()
    return close[~close.index.duplicated()]


def daily_rv(close):
    r = np.log(close).diff().dropna()
    day = r.index.floor("D").tz_localize(None)
    daily = pd.DataFrame({"rv": (r**2).groupby(day).sum(),
                          "ret": r.groupby(day).sum(),
                          "n_bars": r.groupby(day).size()})
    daily.index.name = "date"
    ok = (daily["n_bars"] >= MIN_BARS_PER_DAY) & (daily["rv"] > 0)
    return daily[ok], int((~ok).sum())

In [ ]:
daily = {}
for sym in SYMBOLS:
    cache = DATA_DIR / f"{sym}_daily.parquet"
    if cache.exists():
        daily[sym] = pd.read_parquet(cache)
        continue
    d, n_drop = daily_rv(load_close(sym))
    d.to_parquet(cache)
    daily[sym] = d
    print(f"{sym}: {n_drop} Tage wegen fehlender Bars verworfen")

display(pd.DataFrame({sym: {"Start": d.index[0].date(), "Ende": d.index[-1].date(), "Tage": len(d),
                            "ann. Vola (Median)": round(float(np.sqrt(365 * d["rv"]).median()), 3)}
                      for sym, d in daily.items()}).T)

fig, ax = plt.subplots(figsize=(11, 4))
for sym, d in daily.items():
    ax.plot(np.sqrt(365 * d["rv"]), lw=0.6, label=sym)
ax.set_yscale("log")
ax.set_ylabel("annualisierte Vola  sqrt(365·RV)")
ax.legend()
plt.show()

## 2. Rolling-Origin-Setup

Ein Ursprung $t$ ist der letzte bekannte Tag. Für jeden Ursprung und Horizont $h$ ist das Ziel $RV_{t+h}$. Jeder Experte sieht nur Daten bis einschließlich $t$. Der Split richtet sich nach dem **Zieldatum**, damit kein Label der Trainingsperiode Information aus der Validierungsperiode enthält.

In [ ]:
class RVSeries:
    def __init__(self, sid, df):
        self.sid = sid
        self.dates = df.index
        self.rv = df["rv"].to_numpy(float)
        self.ret = df["ret"].to_numpy(float)
        self.logrv = np.log(self.rv)
        self.origins = np.arange(MIN_HISTORY - 1, len(df) - max(HORIZONS))


def assign_split(target_dates):
    td = pd.to_datetime(target_dates)
    return np.where(td <= pd.Timestamp(TRAIN_END), "train",
                    np.where(td <= pd.Timestamp(VAL_END), "val", "test"))


series = {sym: RVSeries(sym, d) for sym, d in daily.items()}
for s in series.values():
    print(f"{s.sid}: {len(s.origins)} Ursprünge ab {s.dates[s.origins[0]].date()}")

## 3. Experten

Jeder Experte implementiert `forecast(s, origins)` und gibt zwei Arrays auf RV-Skala zurück:

- `mean`: Form `(n_origins, len(HORIZONS))`, der bedingte Erwartungswert als Punktprognose. QLIKE ist für den Mittelwert konsistent (Patton 2011).
- `q`: Form `(n_origins, len(HORIZONS), len(QUANTILES))`, die Quantile.

Die ökonometrischen Modelle prognostizieren in Logs. Ihre Verteilung stammt aus den empirischen Residuen im Schätzfenster, der Mittelwert aus dem Smearing-Schätzer (Duan 1983). Chronos-2 bekommt ebenfalls log-RV. Die Quantile werden per `exp` zurücktransformiert, was exakt ist, weil Quantile unter monotonen Transformationen erhalten bleiben. Den Mittelwert integrieren wir aus den Quantilen.

In [ ]:
_LV = np.asarray(QUANTILES)
_QMASS = (np.r_[_LV[1:], 1.0] - np.r_[0.0, _LV[:-1]]) / 2     # Wahrscheinlichkeitsmasse je Quantil (Trapez)
_QMASS = _QMASS / _QMASS.sum()


def dist_from_log_resid(log_point, resid):
    # Log-Punktprognose + empirische Log-Residuen -> (Mittelwert, Quantile) auf RV-Skala
    resid = resid[np.isfinite(resid)]
    q = np.exp(log_point + np.quantile(resid, _LV))
    mean = np.exp(log_point) * np.mean(np.exp(resid))            # Smearing-Schätzer
    return mean, q


def mean_from_quantiles(q):
    # approximiert E[Y] als Integral der Quantilfunktion
    return q @ _QMASS


class Expert:
    name = "base"

    def forecast(self, s, origins):
        raise NotImplementedError

    @staticmethod
    def _alloc(n):
        return (np.full((n, len(HORIZONS)), np.nan),
                np.full((n, len(HORIZONS), len(QUANTILES)), np.nan))


class RandomWalk(Expert):
    # Baseline: RV_{t+h} = RV_t, Streuung aus historischen h-Schritt-Log-Änderungen
    name = "RW"

    def forecast(self, s, origins):
        L = s.logrv
        mean, q = self._alloc(len(origins))
        for i, t in enumerate(origins):
            lo = max(0, t - EST_WINDOW + 1)
            for j, h in enumerate(HORIZONS):
                resid = L[lo + h: t + 1] - L[lo: t + 1 - h]
                mean[i, j], q[i, j] = dist_from_log_resid(L[t], resid)
        return mean, q

In [ ]:
class HAR(Expert):
    # log-HAR (Corsi 2009): direkte Prognose je Horizont, OLS im rollierenden Fenster
    name = "HAR"

    def forecast(self, s, origins):
        rv = pd.Series(s.rv)
        X = np.column_stack([np.ones(len(rv)), np.log(rv),
                             np.log(rv.rolling(5).mean()), np.log(rv.rolling(22).mean())])
        L = s.logrv
        mean, q = self._alloc(len(origins))
        for i, t in enumerate(origins):
            lo = max(21, t - EST_WINDOW + 1)
            for j, h in enumerate(HORIZONS):
                rows = np.arange(lo, t - h + 1)        # nur Zeilen, deren Ziel u+h <= t schon bekannt ist
                beta, *_ = np.linalg.lstsq(X[rows], L[rows + h], rcond=None)
                resid = L[rows + h] - X[rows] @ beta
                mean[i, j], q[i, j] = dist_from_log_resid(X[t] @ beta, resid)
        return mean, q

In [ ]:
class GARCH(Expert):
    # GARCH(1,1) mit t-Innovationen auf Tagesrenditen. GARCH modelliert die Varianz der
    # Tagesrendite, nicht die RV. Deshalb kalibrieren wir Niveau und Streuung über die
    # Log-Residuen log(RV_u / sigma²_u) im Schätzfenster (1-Schritt-Residuen für alle h).
    name = "GARCH"

    def __init__(self, dist="t", refit_every=GARCH_REFIT_EVERY):
        self.dist, self.refit_every = dist, refit_every

    def forecast(self, s, origins):
        y_all = 100 * s.ret                       # Prozent-Renditen für eine stabile Optimierung
        mean, q = self._alloc(len(origins))
        params = None
        for i, t in enumerate(tqdm(origins, desc=f"GARCH {s.sid}", leave=False)):
            lo = max(0, t - EST_WINDOW + 1)
            am = arch_model(y_all[lo: t + 1], mean="Constant", vol="GARCH", p=1, q=1,
                            dist=self.dist, rescale=False)
            res = None
            if params is None or i % self.refit_every == 0:
                try:
                    res = am.fit(disp="off", show_warning=False,
                                 starting_values=None if params is None else params.to_numpy())
                    params = res.params
                except Exception:
                    res = None
            if res is None:                        # zwischen Refits: alte Parameter auf neue Daten anwenden
                res = am.fix(params)
            sig2_in = np.asarray(res.conditional_volatility) ** 2 / 1e4
            sig2_fc = res.forecast(horizon=max(HORIZONS), reindex=False).variance.to_numpy()[-1] / 1e4
            resid = s.logrv[lo: t + 1] - np.log(sig2_in)
            for j, h in enumerate(HORIZONS):
                mean[i, j], q[i, j] = dist_from_log_resid(np.log(sig2_fc[h - 1]), resid)
        return mean, q

In [ ]:
class Chronos2(Expert):
    # Chronos-2 zero-shot auf log-RV
    name = "Chronos-2"

    def __init__(self, model_id=CHRONOS_MODEL, device=DEVICE,
                 context_length=CHRONOS_CONTEXT, batch_size=CHRONOS_BATCH, chunk=1024):
        from chronos import Chronos2Pipeline
        self.pipe = Chronos2Pipeline.from_pretrained(model_id, device_map=device)
        self.context_length, self.batch_size, self.chunk = context_length, batch_size, chunk

    def forecast(self, s, origins):
        L = s.logrv.astype(np.float32)
        contexts = [L[max(0, t - self.context_length + 1): t + 1] for t in origins]
        preds = []
        for k in tqdm(range(0, len(contexts), self.chunk), desc=f"Chronos-2 {s.sid}", leave=False):
            qs, _ = self.pipe.predict_quantiles(
                contexts[k: k + self.chunk],
                prediction_length=max(HORIZONS),
                quantile_levels=QUANTILES,
                batch_size=self.batch_size,
                context_length=self.context_length,
                cross_learning=False,   # WICHTIG: sonst teilen Kontexte im Batch Information -> Leakage
            )
            preds += [p[0].detach().float().cpu().numpy() for p in qs]     # je (H, nQ)
        log_q = np.stack(preds)[:, [h - 1 for h in HORIZONS], :]
        q = np.sort(np.exp(log_q), axis=-1)                                  # Quantile-Crossing ausschließen
        return mean_from_quantiles(q), q

## 4. Forecasts erzeugen (mit Cache)

Pro Reihe und Modell entsteht eine Parquet-Datei in `results/` im Long-Format: eine Zeile pro Ursprung und Horizont, mit Ziel `y`, `mean` und allen Quantilen. Existiert die Datei schon, wird sie nur geladen. Laufzeiten landen in `results/runtimes.csv`.

**Neues Modell anhängen:** eine Klasse mit `name` und `forecast(s, origins)` schreiben und `run_expert(NeuesModell(), s)` für alle Reihen aufrufen. Alle Auswertungen unten nehmen es automatisch mit.

In [ ]:
def run_expert(expert, s, overwrite=False):
    path = RESULTS_DIR / f"{s.sid}__{expert.name}.parquet"
    if path.exists() and not overwrite:
        return pd.read_parquet(path)
    t0 = time.time()
    mean, q = expert.forecast(s, s.origins)
    runtime = time.time() - t0
    parts = []
    for j, h in enumerate(HORIZONS):
        tgt = s.origins + h
        df = pd.DataFrame({"sid": s.sid, "model": expert.name, "h": h,
                           "origin": s.dates[s.origins], "target_date": s.dates[tgt],
                           "y": s.rv[tgt], "mean": mean[:, j]})
        parts.append(pd.concat([df, pd.DataFrame(q[:, j, :], columns=QCOLS)], axis=1))
    out = pd.concat(parts, ignore_index=True)
    out["split"] = assign_split(out["target_date"])
    out.to_parquet(path)
    with open(RESULTS_DIR / "runtimes.csv", "a") as f:
        f.write(f"{s.sid},{expert.name},{runtime:.1f}\n")
    print(f"{expert.name:>10} | {s.sid}: {runtime:.0f}s")
    return out


def load_forecasts():
    return pd.concat([pd.read_parquet(p) for p in sorted(RESULTS_DIR.glob("*__*.parquet"))],
                     ignore_index=True)

In [ ]:
for s in series.values():
    for expert in [RandomWalk(), HAR(), GARCH()]:
        run_expert(expert, s)

In [ ]:
chronos = Chronos2()          # lädt das Modell von Hugging Face (einmalig)
for s in series.values():
    run_expert(chronos, s)

## 5. Evaluation auf der Validierungsperiode

- **QLIKE** $= \frac{y}{f} - \log\frac{y}{f} - 1$: Hauptmetrik für die Punktprognose, robust gegenüber verrauschten Vola-Proxies (Patton 2011).
- **Pinball-Loss**, gemittelt über alle 21 Quantile: Qualität der ganzen Verteilung. Er ist skalenabhängig, deshalb berichten wir ihn relativ zum Random Walk.
- **Cov90**: Anteil der Ziele im 5 %–95 %-Intervall (Soll: 0,90).

Relative Werte sind pro Reihe und Horizont durch den Random Walk geteilt und über Reihen per geometrischem Mittel aggregiert. **< 1 heißt besser als der Random Walk.**

In [ ]:
def qlike(y, f):
    r = y / f
    return r - np.log(r) - 1


def pinball(y, Q):
    d = y[:, None] - Q
    return np.mean(np.maximum(_LV * d, (_LV - 1) * d), axis=1)


def select_split(df, split):
    if split == "test" and not ALLOW_TEST:
        raise RuntimeError("Die Testperiode ist gesperrt. Erst für die finale Evaluation öffnen (ALLOW_TEST = True).")
    return df[df["split"] == split]


fc = load_forecasts()
fc["qlike"] = qlike(fc["y"].to_numpy(), fc["mean"].to_numpy())
fc["pinball"] = pinball(fc["y"].to_numpy(), fc[QCOLS].to_numpy())
fc["cov90"] = (fc["y"] >= fc["q0.05"]) & (fc["y"] <= fc["q0.95"])
fc["logerr"] = np.log(fc["y"] / fc["mean"])

EVAL_SPLIT = "val"
ev = select_split(fc, EVAL_SPLIT)
print(f"{EVAL_SPLIT}: {len(ev):,} Forecasts,", ev["model"].nunique(), "Modelle")

In [ ]:
agg = (ev.groupby(["sid", "h", "model"])
         .agg(QLIKE=("qlike", "mean"), Pinball=("pinball", "mean"), Cov90=("cov90", "mean"))
         .reset_index())
base = agg.loc[agg["model"] == "RW", ["sid", "h", "QLIKE", "Pinball"]]
agg = agg.merge(base, on=["sid", "h"], suffixes=("", "_RW"))
agg["rel_QLIKE"] = agg["QLIKE"] / agg["QLIKE_RW"]
agg["rel_Pinball"] = agg["Pinball"] / agg["Pinball_RW"]

gmean = lambda x: float(np.exp(np.log(x).mean()))
summary = (agg.groupby(["h", "model"])
              .agg(rel_QLIKE=("rel_QLIKE", gmean), rel_Pinball=("rel_Pinball", gmean), Cov90=("Cov90", "mean"))
              .round(3))
display(summary)
display(agg.pivot_table(index=["sid", "h"], columns="model", values="rel_QLIKE").round(3))

In [ ]:
def dm_test(d, h):
    # Diebold-Mariano auf Loss-Differenzen d, HAC-Varianz nach Newey-West
    d = np.asarray(d, float)
    d = d[np.isfinite(d)]
    T, u = len(d), d - d.mean()
    lag = max(h - 1, 1)
    var = u @ u / T + 2 * sum((1 - k / (lag + 1)) * (u[k:] @ u[:-k]) / T for k in range(1, lag + 1))
    stat = d.mean() / np.sqrt(var / T)
    return stat, 2 * (1 - norm.cdf(abs(stat)))


L_all = ev.pivot_table(index=["sid", "h", "origin"], columns="model", values="qlike")
rows = []
for (sid, h), g in L_all.groupby(level=["sid", "h"]):
    for a, b in [("Chronos-2", "HAR"), ("Chronos-2", "GARCH"), ("HAR", "GARCH")]:
        if a in g and b in g:
            stat, p = dm_test(g[a] - g[b], h)
            rows.append({"sid": sid, "h": h, "Vergleich": f"{a} − {b}", "DM": round(stat, 2), "p": round(p, 3)})
display(pd.DataFrame(rows))   # DM < 0: das erste Modell hat den kleineren QLIKE

## 6. Oracle-Analyse: Lohnt sich ein Gateway?

Der Pool sind die Kandidaten fürs Gateway, der Random Walk bleibt reine Baseline.

- **Oracle:** pro Ursprung rückblickend das beste Modell. Das ist die Obergrenze für jedes Gateway.
- **max. Gewinn %:** Abstand zwischen bestem Einzelmodell und Oracle. Ist er klein, lohnt sich Routing nicht.
- **Gleichgewicht:** der einfache Durchschnitt der Punktprognosen. Diese Baseline muss das Gateway schlagen.
- **Oracle-Beitrag m:** mittleres $\max(0,\ \min_{k \ne m} L_k - L_m)$, also wie viel Modell $m$ gegenüber dem Rest des Pools beiträgt. Nahe null heißt: überflüssig.

In [ ]:
POOL = ["Chronos-2", "HAR", "GARCH"]


def oracle_report(df, pool=POOL):
    key = ["sid", "h", "origin"]
    L = df.pivot_table(index=key, columns="model", values="qlike")[pool].dropna()
    M = df.pivot_table(index=key, columns="model", values="mean")[pool].loc[L.index]
    y = df[df["model"] == pool[0]].set_index(key)["y"].loc[L.index]
    out = []
    for h, Lh in L.groupby(level="h"):
        mean_loss = Lh.mean()
        best = mean_loss.idxmin()
        oracle = Lh.min(axis=1).mean()
        ens = qlike(y.loc[Lh.index].to_numpy(), M.loc[Lh.index].mean(axis=1).to_numpy()).mean()
        row = {"bestes Einzelmodell": best, "QLIKE bestes": mean_loss[best],
               "QLIKE Gleichgewicht": ens, "QLIKE Oracle": oracle,
               "max. Gewinn %": 100 * (mean_loss[best] - oracle) / mean_loss[best]}
        wins = Lh.idxmin(axis=1).value_counts(normalize=True)
        for m in pool:
            row[f"Winrate {m}"] = wins.get(m, 0.0)
        for m in pool:
            others = Lh.drop(columns=m).min(axis=1)
            row[f"Oracle-Beitrag {m}"] = np.maximum(0, others - Lh[m]).mean()
        out.append(pd.Series(row, name=f"h={h}"))
    return pd.concat(out, axis=1)


display(oracle_report(ev))

In [ ]:
# Wie ähnlich sind die Fehler? Hohe Korrelation = wenig Potenzial fürs Routing
E = ev.pivot_table(index=["sid", "h", "origin"], columns="model", values="logerr")[POOL]
for h, Eh in E.groupby(level="h"):
    print(f"h={h}: Korrelation der Log-Fehler log(y/mean)")
    display(Eh.corr().round(2))

In [ ]:
# Wann gewinnt welches Modell? (train + val, h=1): Regimeabhängigkeit sichtbar machen
dev = pd.concat([select_split(fc, "train"), select_split(fc, "val")])
Lw = dev[dev["h"] == 1].pivot_table(index=["sid", "origin"], columns="model", values="qlike")[POOL].dropna()
winner = Lw.idxmin(axis=1)

sids = winner.index.get_level_values("sid").unique()
fig, axes = plt.subplots(len(sids), 1, figsize=(11, 2.6 * len(sids)), sharex=True, squeeze=False)
for ax, sid in zip(axes[:, 0], sids):
    w = winner.xs(sid, level="sid")
    for m in POOL:
        ax.plot((w == m).astype(float).rolling(90, min_periods=30).mean(), label=m, lw=1)
    ax.axvline(pd.Timestamp(TRAIN_END), color="grey", ls="--", lw=0.8)
    ax.set_title(f"{sid}: Anteil Tage als bestes Modell (90-Tage-Fenster, h=1)", fontsize=10)
    ax.set_ylim(0, 1)
axes[0, 0].legend(ncol=len(POOL), fontsize=8)
plt.tight_layout()
plt.show()

## 7. Nächste Schritte

1. **Oracle-Lücke prüfen:** Ist `max. Gewinn %` klein und die Fehlerkorrelation hoch, lohnt sich ein Gateway mit diesem Pool kaum.
2. **Gateway v1:** pro Modell einen Regressor, der dessen relativen Loss vorhersagt, auf `train` trainiert und auf `val` getunt. Features zum Start: Niveau und Trend der log-RV, HAR-Komponenten, Vola der Vola, Rendite der letzten Tage, Wochentag. Stark sind oft auch die Backtest-Fehler der Experten auf den letzten Ursprüngen, sie liegen in den Parquet-Dateien schon vor.
3. **Modelle anhängen:** neue `Expert`-Klasse → `run_expert` → Oracle-Beitrag prüfen.
4. **Erweiterungen:** stündliche RV (Intraday-Saisonalität), weitere Anlageklassen (tägliche OHLC mit Range-Schätzer), Regime-Features.
5. Die **Testperiode** erst ganz am Ende öffnen (`ALLOW_TEST = True`).